# Step 2：数据读取与标准化

本 Notebook 将原始标的、IM 期货和 MO 期权数据转换为后续模块统一使用的 DataFrame。

本模块只负责读取、格式统一、代码解析、日期及期限处理和基础清洗；不计算 Forward、Repo、IV、Greeks、波动率模型或策略结果。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["UNDERLYING_DATA_PATH", "FUTURE_DATA_PATH", "OPTION_DATA_PATH", "UNDERLYING_CODE", "UNDERLYING_PRICE_FIELD", "FUTURE_PREFIX", "FUTURE_PRICE_FIELD", "OPTION_PREFIX", "OPTION_RAW_PRICE_FIELD", "START_DATE", "END_DATE", "TRADING_DAYS_PER_YEAR", "PROCESSED_DATA_PATH", "EXPIRY_DATE_OVERRIDES", "FILTER_ZERO_VOLUME", "SAVE_CSV"]
print(f'01_data_processing.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


In [8]:
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd

# main.ipynb 已执行依赖时直接复用；单独运行本 Notebook 时自动加载依赖。
_required_config = {
    'UNDERLYING_DATA_PATH', 'FUTURE_DATA_PATH', 'OPTION_DATA_PATH',
    'UNDERLYING_CODE', 'UNDERLYING_PRICE_FIELD',
    'FUTURE_PREFIX', 'FUTURE_PRICE_FIELD', 'OPTION_PREFIX',
    'OPTION_RAW_PRICE_FIELD',
    'START_DATE', 'END_DATE', 'TRADING_DAYS_PER_YEAR',
    'PROCESSED_DATA_PATH', 'SAVE_CSV',
    'EXPIRY_DATE_OVERRIDES', 'FILTER_ZERO_VOLUME',
}
_required_functions = {'parse_option_code', 'parse_future_code', 'calculate_expiry_date', 'calculate_tau'}
_ipython = get_ipython() if 'get_ipython' in globals() else None
if not _required_config.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 00_config.ipynb')
    _ipython.run_line_magic('run', './00_config.ipynb')
if not _required_functions.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 02_basic_functions.ipynb')
    _ipython.run_line_magic('run', './02_basic_functions.ipynb')

In [9]:
def _read_csv(path) -> pd.DataFrame:
    """读取 UTF-8 CSV，并对不存在或空文件给出明确错误。"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'数据文件不存在: {path}')
    frame = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    if frame.empty:
        raise ValueError(f'数据文件为空: {path}')
    return frame

def _require_columns(frame: pd.DataFrame, columns: Iterable[str], dataset_name: str) -> None:
    """检查输入数据是否包含指定字段。"""
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f'{dataset_name} 缺少字段: {missing}')

def _filter_date_range(frame: pd.DataFrame, start_date, end_date) -> pd.DataFrame:
    """按闭区间过滤 TRADE_DT 并返回副本。"""
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    if start > end:
        raise ValueError('START_DATE cannot be later than END_DATE')
    return frame.loc[frame['TRADE_DT'].between(start, end)].copy()

def _print_panel_summary(name: str, frame: pd.DataFrame) -> None:
    """打印标准化面板的数据规模、日期范围和前五行。"""
    print(f'{name} loading completed')
    print(f'Rows: {len(frame):,}')
    if frame.empty:
        print('Date range: empty')
    else:
        print(f"Date range: {frame['TRADE_DT'].min().date()} - {frame['TRADE_DT'].max().date()}")
    display(frame.head())

## 标的与期货数据

In [10]:
# 读取并标准化指定指数的每日收盘价数据。
def load_underlying_data(
    path, underlying_code: str, price_field: str, start_date, end_date, verbose: bool = True
) -> pd.DataFrame:
    """读取标的原始数据并生成 TRADE_DT、CODE、CLOSE 三列面板。"""
    raw = _read_csv(path)
    _require_columns(raw, ['CODE', 'DAY', price_field], 'underlying data')
    panel = raw.loc[raw['CODE'].astype(str).eq(underlying_code), ['DAY', 'CODE', price_field]].copy()
    panel.columns = ['TRADE_DT', 'CODE', 'CLOSE']
    panel['TRADE_DT'] = pd.to_datetime(panel['TRADE_DT'], errors='coerce')
    panel['CLOSE'] = pd.to_numeric(panel['CLOSE'], errors='coerce')
    panel = panel.dropna(subset=['TRADE_DT', 'CLOSE'])
    panel = _filter_date_range(panel, start_date, end_date)
    panel = panel.sort_values(['TRADE_DT']).drop_duplicates('TRADE_DT', keep='last').reset_index(drop=True)
    panel['CLOSE'] = panel['CLOSE'].astype(float)
    if panel.empty:
        raise ValueError(f'过滤后没有标的数据: {underlying_code}')
    if verbose:
        _print_panel_summary('Underlying data', panel)
    return panel

# 读取并标准化全部 IM 期货，同时解析合约月份和实际到期日。
def load_future_data(
    path, future_prefix: str, price_field: str, start_date, end_date,
    expiry_overrides: Optional[dict] = None, verbose: bool = True
) -> pd.DataFrame:
    """读取期货原始数据并保留所有可解析的 IM 合约，不筛选主力。"""
    raw = _read_csv(path)
    _require_columns(raw, ['CODE', 'DAY', price_field, 'VOLUME', 'OI'], 'future data')
    prefix_mask = raw['CODE'].astype(str).str.upper().str.startswith(future_prefix.upper())
    panel = raw.loc[prefix_mask, ['DAY', 'CODE', price_field, 'VOLUME', 'OI']].copy()
    panel.columns = ['TRADE_DT', 'CODE', 'CLOSE', 'VOLUME', 'OI']
    panel['TRADE_DT'] = pd.to_datetime(panel['TRADE_DT'], errors='coerce')
    panel['CLOSE'] = pd.to_numeric(panel['CLOSE'], errors='coerce')
    panel['VOLUME'] = pd.to_numeric(panel['VOLUME'], errors='coerce')
    panel['OI'] = pd.to_numeric(panel['OI'], errors='coerce')

    def _safe_parse(code):
        """解析单个期货代码，失败时返回 None 供清洗阶段删除。"""
        try:
            return parse_future_code(code)
        except (TypeError, ValueError):
            return None

    parsed = panel['CODE'].map(_safe_parse)
    panel = panel.loc[parsed.notna()].copy()
    parsed = parsed.loc[parsed.notna()]
    panel['EXPIRY_CODE'] = parsed.map(lambda item: item['expiry_code'])
    panel['EXPIRY'] = panel['EXPIRY_CODE'].map(
        lambda token: pd.Timestamp(calculate_expiry_date(token, expiry_overrides))
    )
    panel = panel.dropna(subset=['TRADE_DT', 'CLOSE', 'VOLUME', 'OI'])
    panel = _filter_date_range(panel, start_date, end_date)
    panel = panel.sort_values(['TRADE_DT', 'CODE']).drop_duplicates(
        ['TRADE_DT', 'CODE'], keep='last'
    ).reset_index(drop=True)
    panel[['CLOSE', 'VOLUME', 'OI']] = panel[['CLOSE', 'VOLUME', 'OI']].astype(float)
    if panel.empty:
        raise ValueError('过滤后没有可用的 IM 期货数据')
    if verbose:
        _print_panel_summary('Future data', panel)
        display(panel.groupby('TRADE_DT')['CODE'].nunique().rename('CONTRACT_COUNT').head())
    return panel

## 期权数据

In [11]:
# 读取并标准化全部 MO 期权，解析合约属性并计算剩余期限 tau。
def load_option_data(
    path, option_prefix: str, raw_price_field: str, start_date, end_date,
    expiry_overrides: Optional[dict] = None, days_per_year: int = 365,
    filter_zero_volume: bool = False, verbose: bool = True
) -> pd.DataFrame:
    """读取 MO 期权并输出统一的日期、代码、到期日、类型、价格和流动性字段。"""
    raw = _read_csv(path)
    source_columns = [
        'TRADE_DT', 'S_INFO_WINDCODE', raw_price_field, 'S_DQ_VOLUME', 'S_DQ_OI'
    ]
    _require_columns(raw, source_columns, 'option data')
    prefix_mask = raw['S_INFO_WINDCODE'].astype(str).str.upper().str.startswith(option_prefix.upper())
    panel = raw.loc[prefix_mask, source_columns].copy()
    panel.columns = ['TRADE_DT', 'CODE', 'PRICE', 'VOLUME', 'OI']
    panel['TRADE_DT'] = pd.to_datetime(panel['TRADE_DT'].astype(str), format='%Y%m%d', errors='coerce')
    for column in ['PRICE', 'VOLUME', 'OI']:
        panel[column] = pd.to_numeric(panel[column], errors='coerce')

    def _safe_parse(code):
        """解析单个期权代码，失败时返回 None 供清洗阶段删除。"""
        try:
            return parse_option_code(code)
        except (TypeError, ValueError):
            return None

    parsed = panel['CODE'].map(_safe_parse)
    panel = panel.loc[parsed.notna()].copy()
    parsed = parsed.loc[parsed.notna()]
    panel['EXPIRY_CODE'] = parsed.map(lambda item: item['expiry_code'])
    panel['TYPE'] = parsed.map(lambda item: item['option_type'])
    panel['STRIKE'] = pd.to_numeric(parsed.map(lambda item: item['strike']), errors='coerce')
    panel['EXPIRY'] = panel['EXPIRY_CODE'].map(
        lambda token: pd.Timestamp(calculate_expiry_date(token, expiry_overrides))
    )
    panel = panel.dropna(subset=['TRADE_DT', 'PRICE', 'STRIKE', 'VOLUME', 'OI', 'EXPIRY'])
    panel = panel.loc[panel['STRIKE'].gt(0)].copy()
    if filter_zero_volume:
        panel = panel.loc[panel['VOLUME'].ne(0)].copy()
    panel = _filter_date_range(panel, start_date, end_date)
    panel = panel.sort_values(['TRADE_DT', 'CODE']).drop_duplicates(
        ['TRADE_DT', 'CODE'], keep='last'
    ).reset_index(drop=True)
    panel['TAU'] = panel.apply(
        lambda row: calculate_tau(row['TRADE_DT'], row['EXPIRY'], days_per_year), axis=1
    ).astype(float)
    panel[['PRICE', 'STRIKE', 'VOLUME', 'OI']] = panel[
        ['PRICE', 'STRIKE', 'VOLUME', 'OI']
    ].astype(float)
    panel = panel[
        ['TRADE_DT', 'CODE', 'EXPIRY', 'TYPE', 'STRIKE', 'PRICE', 'VOLUME', 'OI', 'TAU', 'EXPIRY_CODE']
    ]
    if panel.empty:
        raise ValueError('过滤后没有可用的 MO 期权数据')
    if verbose:
        _print_panel_summary('Option data', panel)
    return panel

## 统一加载、交易日与质量检查

In [12]:
# 构造标的、期货和期权三个市场共同存在的有序交易日列表。
def build_trading_dates(
    underlying_panel: pd.DataFrame, future_panel: pd.DataFrame, option_panel: pd.DataFrame
) -> list[pd.Timestamp]:
    """返回三个标准化面板 TRADE_DT 的有序交集。"""
    common_dates = (
        set(underlying_panel['TRADE_DT'])
        & set(future_panel['TRADE_DT'])
        & set(option_panel['TRADE_DT'])
    )
    if not common_dates:
        raise ValueError('标的、期货和期权不存在共同交易日')
    return sorted(pd.Timestamp(value) for value in common_dates)

# 一次性加载三个市场数据并返回后续模块直接使用的标准化对象。
def load_market_data(verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list]:
    """使用 00_config 中的参数生成 underlying_panel、future_panel、option_panel 和 trading_dates。"""
    underlying = load_underlying_data(
        UNDERLYING_DATA_PATH, UNDERLYING_CODE, UNDERLYING_PRICE_FIELD,
        START_DATE, END_DATE, verbose=verbose,
    )
    futures = load_future_data(
        FUTURE_DATA_PATH, FUTURE_PREFIX, FUTURE_PRICE_FIELD, START_DATE, END_DATE,
        expiry_overrides=EXPIRY_DATE_OVERRIDES, verbose=verbose,
    )
    options = load_option_data(
        OPTION_DATA_PATH, OPTION_PREFIX, OPTION_RAW_PRICE_FIELD, START_DATE, END_DATE,
        expiry_overrides=EXPIRY_DATE_OVERRIDES, days_per_year=TRADING_DAYS_PER_YEAR,
        filter_zero_volume=FILTER_ZERO_VOLUME, verbose=verbose,
    )
    dates = build_trading_dates(underlying, futures, options)
    if verbose:
        print(f'Common trading dates: {len(dates)} ({dates[0].date()} - {dates[-1].date()})')
    return underlying, futures, options, dates

# 汇总三个标准化面板的规模、日期、缺失、重复及期权合约分布。
def run_data_quality_checks(
    underlying_panel: pd.DataFrame, future_panel: pd.DataFrame, option_panel: pd.DataFrame
) -> dict:
    """生成并展示数据处理模块的结构化质量检查结果。"""
    panels = {
        'Underlying': underlying_panel, 'Future': future_panel, 'Option': option_panel
    }
    scale = pd.DataFrame([
        {
            'DATASET': name, 'ROWS': len(frame),
            'START': frame['TRADE_DT'].min(), 'END': frame['TRADE_DT'].max(),
        }
        for name, frame in panels.items()
    ])
    missing = {name: frame.isna().sum().to_dict() for name, frame in panels.items()}
    duplicates = {
        'Underlying': int(underlying_panel.duplicated(['TRADE_DT']).sum()),
        'Future': int(future_panel.duplicated(['TRADE_DT', 'CODE']).sum()),
        'Option': int(option_panel.duplicated(['TRADE_DT', 'CODE']).sum()),
    }
    option_type_counts = option_panel['TYPE'].value_counts().sort_index()
    expiry_counts = option_panel.groupby('EXPIRY')['CODE'].nunique().sort_index()
    daily_option_counts = option_panel.groupby('TRADE_DT')['CODE'].nunique()
    report = {
        'scale': scale, 'missing': missing, 'duplicates': duplicates,
        'option_type_counts': option_type_counts, 'expiry_counts': expiry_counts,
        'daily_option_counts': daily_option_counts,
    }
    print('Data scale and date ranges')
    display(scale)
    print('Missing values')
    display(pd.DataFrame(missing).fillna(0).astype(int))
    print('Duplicate records:', duplicates)
    print('Option type counts')
    display(option_type_counts.rename('COUNT'))
    print(f'Expiry count: {option_panel["EXPIRY"].nunique()}')
    display(expiry_counts.rename('CONTRACT_COUNT'))
    print('Daily option count summary')
    display(daily_option_counts.describe())
    return report

In [13]:
# 将标准化市场面板和交易日列表保存为便于人工查看的 CSV 文件。
def save_processed_data(
    underlying_panel: pd.DataFrame, future_panel: pd.DataFrame,
    option_panel: pd.DataFrame, trading_dates: list, output_path
) -> dict:
    """以 UTF-8 CSV 保存三个标准化 DataFrame 和交易日表，并返回文件路径。"""
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    files = {
        'underlying_panel': output_path / 'underlying_panel.csv',
        'future_panel': output_path / 'future_panel.csv',
        'option_panel': output_path / 'option_panel.csv',
        'trading_dates': output_path / 'trading_dates.csv',
    }
    underlying_panel.to_csv(
        files['underlying_panel'], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d'
    )
    future_panel.to_csv(
        files['future_panel'], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d'
    )
    option_panel.to_csv(
        files['option_panel'], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d'
    )
    pd.DataFrame({'TRADE_DT': pd.to_datetime(trading_dates)}).to_csv(
        files['trading_dates'], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d'
    )
    print('Processed data saved:')
    for name, path in files.items():
        print(f'  {name}: {path}')
    return files

## 执行与输出

运行本单元后生成 `underlying_panel`、`future_panel`、`option_panel`、`trading_dates` 和 `data_quality_report`。

In [14]:
underlying_panel, future_panel, option_panel, trading_dates = load_market_data(verbose=True)
data_quality_report = run_data_quality_checks(underlying_panel, future_panel, option_panel)

print('\nStandardized panel columns')
print('underlying_panel:', underlying_panel.columns.tolist())
print('future_panel:', future_panel.columns.tolist())
print('option_panel:', option_panel.columns.tolist())

processed_data_files = (
    save_processed_data(
        underlying_panel, future_panel, option_panel, trading_dates, PROCESSED_DATA_PATH
    )
    if SAVE_CSV else {}
)

Underlying data loading completed
Rows: 42
Date range: 2026-06-01 - 2026-07-29


,TRADE_DT,CODE,CLOSE
0,2026-06-01,000852.SH,8345.1291
1,2026-06-02,000852.SH,8384.7812
2,2026-06-03,000852.SH,8432.6790
3,2026-06-04,000852.SH,8420.0946
4,2026-06-05,000852.SH,8340.9631


Future data loading completed
Rows: 168
Date range: 2026-06-01 - 2026-07-29


,TRADE_DT,CODE,CLOSE,VOLUME,OI,EXPIRY_CODE,EXPIRY
0,2026-06-01,IM2606.CFE,8305.8,185978.0,219745.0,2606,2026-06-22
1,2026-06-01,IM2607.CFE,8235.2,10555.0,16869.0,2607,2026-07-17
2,2026-06-01,IM2609.CFE,8081.8,58978.0,141306.0,2609,2026-09-18
3,2026-06-01,IM2612.CFE,7880.4,22039.0,66830.0,2612,2026-12-18
4,2026-06-02,IM2606.CFE,8324.6,175916.0,210218.0,2606,2026-06-22


TRADE_DT
2026-06-01    4
2026-06-02    4
2026-06-03    4
2026-06-04    4
2026-06-05    4
Name: CONTRACT_COUNT, dtype: int64

...

Expiry count: 8


EXPIRY
2026-06-22    82
2026-07-17    64
2026-08-21    74
2026-09-18    76
2026-10-16    38
2026-12-18    40
2027-03-19    40
2027-06-18    38
Name: CONTRACT_COUNT, dtype: int64

Daily option count summary


count     42.000000
mean     289.571429
std       15.241380
min      266.000000
25%      271.000000
50%      294.000000
75%      304.000000
max      306.000000
Name: CODE, dtype: float64


Standardized panel columns
underlying_panel: ['TRADE_DT', 'CODE', 'CLOSE']
future_panel: ['TRADE_DT', 'CODE', 'CLOSE', 'VOLUME', 'OI', 'EXPIRY_CODE', 'EXPIRY']
option_panel: ['TRADE_DT', 'CODE', 'EXPIRY', 'TYPE', 'STRIKE', 'PRICE', 'VOLUME', 'OI', 'TAU', 'EXPIRY_CODE']
Processed data saved:
  underlying_panel: /Users/mac/Desktop/实习/code8.10/8.10/outputs/processed_data/underlying_panel.csv
  future_panel: /Users/mac/Desktop/实习/code8.10/8.10/outputs/processed_data/future_panel.csv
  option_panel: /Users/mac/Desktop/实习/code8.10/8.10/outputs/processed_data/option_panel.csv
  trading_dates: /Users/mac/Desktop/实习/code8.10/8.10/outputs/processed_data/trading_dates.csv
